**Changes on 30 Oct (P1 + Claude), not yet run.** Run order: first re-run 01's `metrics.py` cell and `[S3 tuning]` (new `tuned.yaml`), then *Run all* here.
- New `[S6 Cell B2]`: Poisson-Gaussian grid -> `results/synthetic_pg_test.csv`.
- `[S6 Cell C]` unchanged, but must be re-run for correct MTF50 values.
- `[S6 Cell D]`: tests both noise models and saves `results/wilcoxon.csv`.
- `[S7 Cell A]`: all 8 test crops x 3 seeds, saved to `results/psf_mismatch.csv`.
- `[S7 Cell B]`: all 8 crops x 3 linear methods, saved to `results/boundary.csv` + `figures/boundary_error_map.png`.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import os, sys
ROOT = '/content/drive/MyDrive/DSP_Deconv'
for d in ['lib', 'data/raw', 'data/processed', 'results', 'figures', 'configs']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
if f'{ROOT}/lib' not in sys.path:
    sys.path.insert(0, f'{ROOT}/lib')
print(os.listdir(ROOT))

['lib', 'data', 'results', 'figures', 'configs', 'test.ipynb', '00_setup_data.ipynb', '01_forward_baselines.ipynb', '02_proposed.ipynb', '03_experiments.ipynb']


In [13]:
!pip -q install scikit-image tifffile pyyaml

import sys
import importlib

if "imp" not in sys.modules:
    sys.modules["imp"] = importlib

# Enable autoreload
%load_ext autoreload
%autoreload 2

import numpy as np, scipy, skimage, matplotlib.pyplot as plt
print('numpy', np.__version__, '| scipy', scipy.__version__, '| skimage', skimage.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
numpy 2.1.3 | scipy 1.16.3 | skimage 0.25.2


In [14]:
import yaml
CFG_PATH = f'{ROOT}/configs/params.yaml'
if not os.path.exists(CFG_PATH):
    cfg = dict(
        seeds=[0, 1, 2, 3, 4],
        psf_sigmas=[1.5, 2.5, 4.0],            # Gaussian PSF width, pixels
        gauss_noise=[0.005, 0.01, 0.02, 0.05],  # std on a 0-1 image
        poisson_peaks=[30, 100, 300],           # photons at the brightest pixel
        crop=256,                               # work on 256x256 crops for speed
        eps_band=1e-3,                          # |H|^2 below this = 'noise-only' band
    )
    yaml.safe_dump(cfg, open(CFG_PATH, 'w'))
cfg = yaml.safe_load(open(CFG_PATH))
cfg

{'crop': 256,
 'eps_band': 0.001,
 'gauss_noise': [0.005, 0.01, 0.02, 0.05],
 'poisson_peaks': [30, 100, 300],
 'psf_sigmas': [1.5, 2.5, 4.0],
 'seeds': [0, 1, 2, 3, 4]}

In [15]:
# [S6 Cell A, in 03_experiments.ipynb] test images + method table
import itertools, time, yaml, pandas as pd
from forward import *; from data import *; from baselines import *; from proposed import *; from metrics import *
tuned = yaml.safe_load(open(f'{ROOT}/configs/tuned.yaml'))      # {'nsr': ..., 'rl_iter': ...} from dev set

test_imgs = []
for subset in ['Confocal_BPAE_B', 'TwoPhoton_MICE']:
    gt, _ = fmd_pair(subset, 19)
    for q, crop in enumerate(quadrants(norm_pair(gt)[0])):
        test_imgs.append((f'{subset}_q{q}', crop))

METHODS = {
    'inverse':       lambda y, psf, c: inverse(y, psf),
    'wiener_const':  lambda y, psf, c: wiener_const(y, psf, tuned['nsr']),
    'tikhonov_gcv':  lambda y, psf, c: tikhonov_gcv(y, psf),
    'rl':            lambda y, psf, c: rl(y, psf, tuned['rl_iter']),
    'unsup_wiener':  lambda y, psf, c: unsup_wiener(y, psf),
    'proposed':      lambda y, psf, c: adaptive_wiener(y, psf),
    'oracle_wiener': lambda y, psf, c: wiener_oracle(y, psf, c['x'], c['sigma']),
}

In [16]:
# [S6 Cell B] Gaussian-noise grid (Poisson-Gaussian: same loop with add_poisson_gaussian and cfg['poisson_peaks'])
rows = []
for (name, x), s_psf, s_n, seed in itertools.product(test_imgs, cfg['psf_sigmas'], cfg['gauss_noise'], cfg['seeds']):
    rng = np.random.default_rng(seed)
    psf = gaussian_psf(psf_size(s_psf), s_psf)
    y = add_gaussian(blur(x, psf), s_n, rng)
    for m, f in METHODS.items():
        t0 = time.perf_counter(); xh = f(y, psf, dict(x=x, sigma=s_n)); dt = time.perf_counter() - t0
        xc = np.clip(xh, 0, 1)
        rows.append(dict(img=name, psf=s_psf, noise=s_n, seed=seed, method=m,
                         psnr=psnr(x, xc), ssim=ssim(x, xc), lsd=lsd(x, xc), ms=1e3 * dt))
df = pd.DataFrame(rows)
df.to_csv(f'{ROOT}/results/synthetic_gauss_test.csv', index=False)
df.groupby(['psf', 'noise', 'method'])[['psnr', 'ssim']].agg(['mean', 'std']).round(2)

psnr        ssim      
                          mean   std  mean   std
psf noise method                                
1.5 0.005 inverse         3.37  0.09  0.00  0.00
          oracle_wiener  34.79  1.54  0.90  0.03
          proposed       34.47  1.53  0.88  0.04
          rl             31.73  1.51  0.92  0.04
          tikhonov_gcv   34.53  1.51  0.90  0.04
...                        ...   ...   ...   ...
4.0 0.050 proposed       27.31  1.86  0.68  0.07
          rl             25.97  1.60  0.70  0.05
          tikhonov_gcv   27.29  1.83  0.71  0.04
          unsup_wiener   20.52  0.36  0.18  0.02
          wiener_const   27.13  1.75  0.68  0.03

[84 rows x 4 columns]

In [ ]:
# [S6 Cell B2] Poisson-Gaussian grid (photon-limited microscopy noise); read-noise std fixed at 0.01
READ = 0.01
pg_rows = []
for (name, x), s_psf, peak, seed in itertools.product(test_imgs, cfg['psf_sigmas'], cfg['poisson_peaks'], cfg['seeds']):
    rng = np.random.default_rng(seed)
    psf = gaussian_psf(psf_size(s_psf), s_psf)
    yb = blur(x, psf)
    y = add_poisson_gaussian(yb, peak, READ, rng)
    sigma_eff = np.sqrt(np.clip(yb, 0, None).mean() / peak + READ ** 2)   # average noise std, used by the oracle only
    for m, f in METHODS.items():
        if m == 'inverse': continue
        t0 = time.perf_counter(); xh = f(y, psf, dict(x=x, sigma=sigma_eff)); dt = time.perf_counter() - t0
        xc = np.clip(xh, 0, 1)
        pg_rows.append(dict(img=name, psf=s_psf, peak=peak, seed=seed, sigma_eff=sigma_eff, method=m,
                            psnr=psnr(x, xc), ssim=ssim(x, xc), lsd=lsd(x, xc), ms=1e3 * dt))
pg = pd.DataFrame(pg_rows)
pg.to_csv(f'{ROOT}/results/synthetic_pg_test.csv', index=False)
pg.groupby(['psf', 'peak', 'method'])[['psnr', 'ssim']].agg(['mean', 'std']).round(2)

In [ ]:
# [S6 Cell C] edge phantom: resolution, ringing, noise amplification per method
# Re-run after the 30 Oct metrics.py fix: the old MTF50 column was wrong
edge_rows = []
for s_n, seed in itertools.product(cfg['gauss_noise'], cfg['seeds']):
    x = edge_phantom(); psf = gaussian_psf(psf_size(2.5), 2.5)
    y = add_gaussian(blur(x, psf), s_n, np.random.default_rng(seed))
    for m, f in METHODS.items():
        if m == 'inverse': continue
        e = edge_metrics(f(y, psf, dict(x=x, sigma=s_n)))
        edge_rows.append(dict(noise=s_n, seed=seed, method=m, noise_amp=e['flat_std'] / s_n, **e))
pd.DataFrame(edge_rows).to_csv(f'{ROOT}/results/edge_test.csv', index=False)

In [ ]:
# [S6 Cell D] is the proposed method significantly better? paired Wilcoxon test over images x settings x seeds
from scipy.stats import wilcoxon
tests = []
for label, d, keys in [('gaussian', df, ['img', 'psf', 'noise', 'seed']), ('poisson_gaussian', pg, ['img', 'psf', 'peak', 'seed'])]:
    a = d[d.method == 'proposed'].sort_values(keys).psnr.values
    for base in ['wiener_const', 'tikhonov_gcv', 'rl', 'unsup_wiener']:
        b = d[d.method == base].sort_values(keys).psnr.values
        tests.append(dict(noise_model=label, baseline=base, n=len(a), median_gain_db=np.median(a - b),
                          mean_gain_db=np.mean(a - b), frac_proposed_wins=np.mean(a > b), p_value=wilcoxon(a, b).pvalue))
tests = pd.DataFrame(tests)
tests.to_csv(f'{ROOT}/results/wilcoxon.csv', index=False)
tests.round(4)

In [ ]:
# [S7 Cell A] PSF mismatch: deconvolve with a PSF that is too narrow or too wide (8 test crops x 3 seeds)
mis = []
s_true, s_n = 2.5, 0.01
for (name, x), seed in itertools.product(test_imgs, cfg['seeds'][:3]):
    y = add_gaussian(blur(x, gaussian_psf(psf_size(s_true), s_true)), s_n, np.random.default_rng(seed))
    for dlt in [-0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3]:
        s_used = s_true * (1 + dlt); psf_used = gaussian_psf(psf_size(s_used), s_used)
        for m in ['wiener_const', 'tikhonov_gcv', 'rl', 'proposed']:
            xh = np.clip(METHODS[m](y, psf_used, {}), 0, 1)
            mis.append(dict(img=name, seed=seed, error_pct=100 * dlt, method=m, psnr=psnr(x, xh)))
mis = pd.DataFrame(mis)
mis.to_csv(f'{ROOT}/results/psf_mismatch.csv', index=False)
mis.pivot_table(index='error_pct', columns='method', values='psnr').plot(marker='o', ylabel='mean PSNR (dB)')

In [ ]:
# [S7 Cell B] boundaries: blur with realistic 'reflect' borders, then deconvolve with and without mirror padding
psf = gaussian_psf(psf_size(2.5), 2.5)
bnd = []
for name, x in test_imgs:
    y = add_gaussian(blur(x, psf, boundary='reflect'), 0.01, np.random.default_rng(0))
    for m in ['wiener_const', 'tikhonov_gcv', 'proposed']:
        f = lambda im, p: METHODS[m](im, p, {})
        bnd.append(dict(img=name, method=m,
                        psnr_plain=psnr(x, np.clip(f(y, psf), 0, 1)),
                        psnr_padded=psnr(x, np.clip(padded(f, y, psf), 0, 1))))
bnd = pd.DataFrame(bnd)
bnd.to_csv(f'{ROOT}/results/boundary.csv', index=False)

x = test_imgs[0][1]
y = add_gaussian(blur(x, psf, boundary='reflect'), 0.01, np.random.default_rng(0))
plt.imshow(np.abs(np.clip(adaptive_wiener(y, psf), 0, 1) - x), cmap='inferno'); plt.title('error map, no padding'); plt.colorbar()
plt.savefig(f'{ROOT}/figures/boundary_error_map.png', dpi=200); plt.show()
bnd.groupby('method')[['psnr_plain', 'psnr_padded']].agg(['mean', 'std']).round(2)